In [2]:
# !pip install pandas
import pandas as pd
import re
import pickle
from datetime import datetime
from bertopic import BERTopic
import matplotlib.pyplot as p
import networkx as nx
import requests
import numpy as np
# import networks_layouts
from matplotlib import colors
import pickle

import ast
def convert_str_to_list(x):
    try:
        return ast.literal_eval(x, '|')
    except:
        return None
def read_pickle(file_name):
    '''Function to read pickles faster'''
    pickle_in = open(file_name, 'rb')
    variable = pickle.load(pickle_in)
    return variable

def write_pickle(variable, file_name):
    '''Function to write pickles faster'''
    pickle_out = open(file_name, 'wb')
    pickle.dump(variable, pickle_out)

#Move all files of one folder to another one:
from os import listdir
from os.path import isfile, join
import os
import shutil

# From here starts the text analyses:

In [3]:
#!pip install nltk
from sentence_transformers import SentenceTransformer
import sys
sys.path.insert(1, '/Users/anajaramillo/.local/lib/python3.13/site-packages')

from numpy.typing import NDArray
import torch
from torch import tensor

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

def split_into_sentences(text):
  """Splits a text into sentences using NLTK’s sentence tokenizer."""
  from nltk.tokenize import sent_tokenize
  return sent_tokenize(text)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/anajaramillo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/anajaramillo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
%%time
from scipy.spatial import distance
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import matplotlib.pyplot as plt
vectorizer = CountVectorizer()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6819.79it/s]


CPU times: user 179 ms, sys: 203 ms, total: 382 ms
Wall time: 1.49 s


In [5]:
#Abstract preprocessing
nltk.download('stopwords')
from nltk.corpus import stopwords
from collections import Counter
import re
stop_words = stopwords.words('english')
stopwords_dict = Counter(stop_words)

def pre_process_text(text):
    dlist = re.sub(r'https?:\/\/.*[\r\n]*', ' ', str(text), flags=re.MULTILINE) #removes links
    dlist = re.sub(r'[^A-Za-z \']', ' ', str(dlist)).lower()   #lowercase
    dlist = re.sub(r'[,"!?:@\$%_]+', ' ', str(dlist), flags=re.MULTILINE)
    dlist = re.sub(r'\.+', ' ', str(dlist), flags=re.MULTILINE) #remove multiple dots
    dlist = re.sub(r'\?+', ' ', str(dlist), flags=re.MULTILINE) #remove multiple ?
    dlist = re.sub(r'\!+', ' ', str(dlist), flags=re.MULTILINE) #remove multiple !
    dlist = re.sub(r"'", '', str(dlist), flags=re.MULTILINE)
    dlist = re.sub(r"\s+"," ", str(dlist), flags=re.MULTILINE) #remove multiple spaces
    dlist = re.sub(r'[0-9]+', '', str(dlist)) #remove numbers from the string
    dlist = ' '.join([word for word in dlist.split() if word not in stopwords_dict])
    return dlist.upper()


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anajaramillo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
List_translated_abstract = read_pickle('List_translated_abstracts/List_translated_abstract.pkl')
Filtered_DataFrame_out = read_pickle('Filtered_DataFrame_out.pkl')

In [7]:
%%time
english_abstract = []
for i in Filtered_DataFrame_out.index:
    id_paper = Filtered_DataFrame_out.id[i]
    try:
        english_abstract += [List_translated_abstract[id_paper]]
    except:
        if Filtered_DataFrame_out.detected_language[i] == 'en':
            english_abstract += [Filtered_DataFrame_out.abstract[i]]
        else:
            english_abstract += ['']
        pass
            
Filtered_DataFrame_out['english_abstract'] = english_abstract       

CPU times: user 1.22 s, sys: 7.13 ms, total: 1.22 s
Wall time: 1.23 s


In [12]:
Filtered_DataFrame_out_filt = Filtered_DataFrame_out[Filtered_DataFrame_out.english_abstract!='']
Filtered_DataFrame_out_filt = Filtered_DataFrame_out_filt.drop_duplicates('abstract')

In [16]:
Filtered_DataFrame_out_filt = Filtered_DataFrame_out_filt[~Filtered_DataFrame_out_filt['primary_location.source.id'].isna()]


In [17]:
Filtered_DataFrame_out_filt['primary_location.source.id'] = [i.replace('https://openalex.org/','') for i in Filtered_DataFrame_out_filt['primary_location.source.id']]

In [10]:
def Concepts_search_and_generate_DF_over_all(Concepts, Filtered_DataFrame_out_filt):
    Dict_concepts_keys = {}
    Keys_list = ['english_abstract', 'primary_topic.display_name']
    Dict_concepts_keys = {}
    
    DF_with_key = {Filtered_DataFrame_out_filt.id[i]:';'.join([str(Filtered_DataFrame_out_filt[k][i]).upper() for k in Keys_list]) for i in Filtered_DataFrame_out_filt.index }
    
    for concept in Concepts:
        Dict_concepts_keys[concept] = [i for i,v in DF_with_key.items() if concept.upper() in v]
    
    return pd.DataFrame({'concept':[i for i in Dict_concepts_keys.keys()],
                  'Number of papers': [str(len(v))+' ('+str(np.round(len(v)/Filtered_DataFrame_out_filt.shape[0],3))+')' for i,v in Dict_concepts_keys.items()]})


In [18]:
Dict_concepts_keys = {}
for key in ['english_abstract', 'primary_topic.display_name']:
    Dict_concepts_keys[key] = {}

    DF_with_key = Filtered_DataFrame_out_filt[~Filtered_DataFrame_out_filt[key].isnull()]
    
    for concept in Concepts_of_peace:
        Dict_concepts_keys[key][concept] = DF_with_key[DF_with_key[key].apply(lambda x: concept.upper() in x.upper())].shape[0]
    
pd.DataFrame(Dict_concepts_keys)

,english_abstract,primary_topic.display_name
negative peace,158,0
positive peace,337,0
restorative justice,318,0
state building,74,0
reconciliation,2181,0
security,6846,4217
community,5482,43


In [21]:
#downloaded from here: https://ourworldindata.org/grapher/world-regions-un2?overlay=download-data (the regions of the world)
World_regions = pd.read_csv('world-regions-un2.csv')
World_regions_M49 = pd.read_csv('RegionM49.csv')
Dict_region_per_country = {World_regions_M49['codeforiati:iso-alpha-2-code'][i]:World_regions_M49['codeforiati:sub-region-name'][i] for i in World_regions_M49.index}
Dict_region_per_country['NA'] = 'Sub-Saharan Africa'
Dict_region_per_country['AN'] = 'Latin America and the Caribbean'
Dict_region_per_country['TW'] = 'Eastern Asia'
Country_codes = pd.read_csv('Country_codes.csv', delimiter=',')
Country_GDP = pd.read_csv('API_NY.GDP.PCAP.CD_DS2_en_csv_v2_4770417.csv')

In [23]:
#Read all the journal information already downloaded:
Dict_journal_information = read_pickle('Dict_journal_information_multilingual.pkl')

In [24]:
#Count number of journals per contry:
Dict_journals_per_country = {}
Dict_journals_per_country_count = {}
for i in Dict_journal_information.keys():
    try:
        Dict_journals_per_country_count[Dict_journal_information[i]['country_code']] += 1
        Dict_journals_per_country[Dict_journal_information[i]['country_code']] += [i]
    except:
        Dict_journals_per_country_count[Dict_journal_information[i]['country_code']] = 1
        Dict_journals_per_country[Dict_journal_information[i]['country_code']] = [i]
        
        pass

In [25]:
#Bring the country that belong to each region:
Dict_countries_regions = {}
for k in Dict_journals_per_country_count.keys():
    try:
        Dict_countries_regions[World_regions_M49[World_regions_M49['codeforiati:iso-alpha-2-code'] == k]['codeforiati:sub-region-name'].iloc[0]] += [k] 
    except:
        try:
            Dict_countries_regions[World_regions_M49[World_regions_M49['codeforiati:iso-alpha-2-code'] == k]['codeforiati:sub-region-name'].iloc[0]] = [k] 
        except:
            pass
        pass

In [76]:
Dict_papers_per_region = {}

for i in Filtered_DataFrame_out_filt.index:
    paper = Filtered_DataFrame_out_filt.id[i]
    journal = Filtered_DataFrame_out_filt['primary_location.source.id'][i]
    for region in Dict_countries_regions.keys():
        for country in Dict_countries_regions[region]:
            if journal in Dict_journals_per_country[country]:
                try:
                    Dict_papers_per_region[region] += [paper]
                except:
                    Dict_papers_per_region[region] = [paper]
                    pass
                    
Dict_number_papers_per_region = {}

for k in Dict_papers_per_region.keys():
    Dict_number_papers_per_region[k] = int(len(Dict_papers_per_region[k]))                       

In [77]:
Dict_number_papers_per_region

{'Eastern Europe': 1379,
 'Southern Asia': 1161,
 'Northern Africa': 135,
 'Western Asia': 419,
 'Latin America and the Caribbean': 3194,
 'Southern Europe': 1158,
 'Western Europe': 1187,
 'Northern America': 2145,
 'Northern Europe': 2222,
 'South-eastern Asia': 2104,
 'Polynesia': 9,
 'Eastern Asia': 586,
 'Australia and New Zealand': 123,
 'Sub-Saharan Africa': 740,
 'Micronesia': 15,
 'Central Asia': 22}

In [27]:
def Concepts_search_and_generate_DF_with_regions(Concepts, Filtered_DataFrame_out_filt, Dict_countries_regions,Dict_number_papers_per_region, separated=False):
    Dict_concepts_keys = {}
    Keys_list = ['english_abstract', 'primary_topic.display_name']
    Dict_concepts_keys = {}
    
    DF_with_key = {Filtered_DataFrame_out_filt.id[i]:';'.join([str(Filtered_DataFrame_out_filt[k][i]).upper() for k in Keys_list]) for i in Filtered_DataFrame_out_filt.index }
    
    for concept in Concepts:
        list_concepts = concept.split(' ')
        if len(list_concepts) == 1:
            Dict_concepts_keys[concept] = [i for i,v in DF_with_key.items() if concept.upper() in v]
        else:
            if separated == True:
                Dict_concepts_keys[concept] = [i for i,v in DF_with_key.items() if (list_concepts[0].upper() in v) and (list_concepts[1].upper() in v)]
            else:
                Dict_concepts_keys[concept] = [i for i,v in DF_with_key.items() if (concept.upper() in v) or (''.join(list_concepts).upper() in v) or ('-'.join(list_concepts).upper() in v)]
                
    
    Dict_concept_region_papers = {}

    for concept in Concepts:
        Dict_concept_region_papers[concept] = {}
        for paper in Dict_concepts_keys[concept]:
            journal = Filtered_DataFrame_out_filt[Filtered_DataFrame_out_filt.id==paper]['primary_location.source.id'].iloc[0]
            for region in Dict_countries_regions.keys():
                for country in Dict_countries_regions[region]:
                    if journal in Dict_journals_per_country[country]:
                        try:
                            Dict_concept_region_papers[concept][region] += [paper]
                        except:
                            Dict_concept_region_papers[concept][region] = [paper]
                            pass
    
    Dict_concept_region_number_of_papers = {}
    
    for k in Dict_concept_region_papers.keys():
        Dict_concept_region_number_of_papers[k] = {}
        for v in Dict_concept_region_papers[k].keys():
            n_papers_c_r = len(Dict_concept_region_papers[k][v])
            p_papers_r =  Dict_number_papers_per_region[v]
            Dict_concept_region_number_of_papers[k][v] = str(n_papers_c_r)+ ' (' + str(np.round(n_papers_c_r/p_papers_r,2)) + ')'
    
    return pd.DataFrame(Dict_concept_region_number_of_papers).sort_index()

In [ ]:
Dictionary_of_terms = pd.read_table('Dict_of_terms/Dictionary_of_terms_liberal_dominant_paradigms.csv',delimiter=';',header=0)
Filtered_Data = Semantic_search(Filtered_DataFrame_out_filt.loc[:10], Dictionary_of_terms)


0


,id,abstract,authorships.author.display_name,authorships.author.id,authorships.author.orcid,authorships.countries,authorships.is_corresponding,best_oa_location.license,cited_by_count,corresponding_institution_ids,...,Appearance_term_Peacebuilding,Probability_term_Peacemaking,Probability_string_term_Peacemaking,Appearance_term_Peacemaking,Probability_term_Reconciliation,Probability_string_term_Reconciliation,Appearance_term_Reconciliation,Probability_term_Development,Probability_string_term_Development,Appearance_term_Development
1,https://openalex.org/W2790530886,Στην εισαγωγή επιχειρείται η προσέγγιση των βα...,Στέργιος Μαλάκης,https://openalex.org/A5032341241,NaN,NaN,True,NaN,0,NaN,...,Yes,"[0.13, 0.12, 0.14, 0.22, 0.09, 0.24, 0.15, 0.1...",<=0.4,Yes,"[0.14, 0.2, 0.26, 0.3, 0.14, 0.31, 0.14, 0.2, ...",<=0.4,Yes,"[0.03, 0.01, 0.17, 0.2, 0.13, 0.3, 0.05, 0.16,...",<=0.4,Yes
3,https://openalex.org/W7116735838,EN This short position paper argues that Gener...,František Havel,https://openalex.org/A5120962082,NaN,NaN,True,cc-by,0,NaN,...,Yes,"[0.33, 0.3, 0.49, 0.33, 0.31, 0.47, 0.35]",>0.4,Yes,"[0.38, 0.41, 0.4, 0.4, 0.42, 0.4, 0.38]",>0.4,Yes,"[0.17, 0.14, 0.09, 0.16, 0.14, 0.07, 0.06]",<=0.4,Yes
5,https://openalex.org/W4391831950,"Článek zkoumá komplexní vztah mezi irénismem, ...",Libor Pavera,https://openalex.org/A5093931776,NaN,NaN,True,cc-by-sa,0,NaN,...,Yes,"[0.25, 0.31, 0.12, 0.2, 0.0, 0.16, 0.09]",<=0.4,Yes,"[0.41, 0.4, 0.32, 0.18, 0.17, 0.34, 0.2]",>0.4,Yes,"[0.06, 0.04, 0.13, 0.13, 0.02, 0.14, 0.04]",<=0.4,Yes
8,https://openalex.org/W4392020227,آیات قرآن متناسب با شرایط عصر نزول که 23 سال ب...,محیا پوریزدانپناه کرمانی|زهره اخوان مقدم,https://openalex.org/A5093976060|https://opena...,|,|,True|False,cc-by-sa,0,NaN,...,Yes,"[0.33, 0.06, 0.33, 0.37, 0.19, -0.02]",<=0.4,Yes,"[0.32, 0.12, 0.28, 0.42, 0.15, 0.04]",>0.4,Yes,"[-0.01, -0.05, 0.08, -0.03, -0.05, 0.04]",<=0.4,Yes


In [58]:
Filtered_DataFrame_out_filt_extra[Filtered_DataFrame_out_filt_extra.id=='https://openalex.org/W4392020227'].Probability_term_Reconciliation

8        [0.31]
17215    [0.31]
Name: Probability_term_Reconciliation, dtype: object

# 2. Search the terms based on the dictionary:


In [302]:
from sentence_transformers import SentenceTransformer

# Loading the model 
model = SentenceTransformer('all-MiniLM-L6-v2')

def Compare_Text_with_terms(text_1, Dictionary_of_terms):

    Probability_term = {Term.upper():[] for Term in Dictionary_of_terms.Term}
    Appearance_term = {Term.upper():[] for Term in Dictionary_of_terms.Term}
    
    text = pre_process_text(text_1)
    sentences = split_into_sentences(text_1)

    # Generating vector embeddings from the list
    embeddings = model.encode(sentences, show_progress_bar=False)
    
    for index in Dictionary_of_terms.index:
        term,user_query = Dictionary_of_terms['Term'][index].upper(),Dictionary_of_terms['Definition'][index].upper()
        user_query = split_into_sentences(user_query) 
        query_embedding = model.encode([pre_process_text(user_query)])[0]
        List_similarities = []
        for sentence in embeddings:
            similarity = np.dot(
              np.array(query_embedding), 
              np.array(sentence)
            ) / (
              np.linalg.norm(np.array(query_embedding)) * 
              np.linalg.norm(sentence)
            )
            List_similarities  += [np.round(similarity,2)]

        Probability_term[term] = List_similarities

        if ' ' in term:
            list_concepts = term.split(' ')
        elif '-' in term:
            list_concepts = term.split('-')
        else:
            list_concepts = [term]
            
        if len(list_concepts) == 1:
            if term.upper() in text.upper():
                Appearance_term[term] = 'Yes'
            else:
                Appearance_term[term] = 'No'    
        else:
            if term in text_1:
                Appearance_term[term] = 'Yes'
            elif ''.join(list_concepts).upper() in text_1:
                Appearance_term[term] = 'Yes'
            elif '-'.join(list_concepts).upper() in text_1:
                Appearance_term[term] = 'Yes'
            elif ' '.join(list_concepts).upper() in text_1:
                Appearance_term[term] = 'Yes'
            else:
                appears = []
                for concept in list_concepts:
                    if concept in text_1:
                        appears +=[True]
                    else:
                        appears +=[False]
                if False not in appears:  
                    Appearance_term[term] = 'Yes'
                else:
                    Appearance_term[term] = 'No'

    return Probability_term, Appearance_term
    
        

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6432.31it/s]


In [471]:
from sentence_transformers import SentenceTransformer

# Loading the model 

def Compare_Text_with_terms_appearance(text_1, Dictionary_of_terms):

    Appearance_term = {Term.upper():[] for Term in Dictionary_of_terms.Term}
    
    # Generating vector embeddings from the list    
    for index in Dictionary_of_terms.index:
        term = Dictionary_of_terms['Term'][index].upper()       

        if ' ' in term:
            list_concepts = term.split(' ')
        elif '-' in term:
            list_concepts = term.split('-')
        else:
            list_concepts = [term]
            
        if len(list_concepts) == 1:
            if ' '+term+' '.upper() in text_1.upper() or '.'+term+' '.upper() in text_1.upper()\
                or ','+term+' '.upper() in text_1.upper() or ' '+term+'.'.upper() in text_1.upper()\
                or ' '+term+','.upper() in text_1.upper() or '-'+term+' '.upper() in text_1.upper()\
                or ' '+term+'-'.upper() in text_1.upper() or ' '+term+'-'.upper() in text_1.upper():
                Appearance_term[term] = 'Yes'
            else:
                Appearance_term[term] = 'No'    
        else:
            if term in text_1:
                Appearance_term[term] = 'Yes'
            elif ''.join(list_concepts).upper() in text_1:
                Appearance_term[term] = 'Yes'
            elif '-'.join(list_concepts).upper() in text_1:
                Appearance_term[term] = 'Yes'
            elif ' '.join(list_concepts).upper() in text_1:
                Appearance_term[term] = 'Yes'
            else:
            #     appears = []
            #     for concept in list_concepts:
            #         if concept in text_1:
            #             appears +=[True]
            #         else:
            #             appears +=[False]
            #     if False not in appears:  
            #         Appearance_term[term] = 'Yes'
            #     else:
                Appearance_term[term] = 'No'

    return Appearance_term
    
        

In [443]:
def Semantic_search_fast(Filtered_DataFrame_out_filt, Dictionary_of_terms):
    Dict_concepts_keys = {}
    Keys_list = ['english_abstract', 'primary_topic.display_name']
    Dict_concepts_keys = {}
    
    DF_with_key = {Filtered_DataFrame_out_filt.id[i]:';'.join([str(Filtered_DataFrame_out_filt[k][i]).upper()+'.' for k in Keys_list]) for i in Filtered_DataFrame_out_filt.index }

    Probability_term, Appearance_term = {}, {}
    for i,k in enumerate(DF_with_key.keys()):
        if i%1000==0:
            print(i)
        text = DF_with_key[k]
        Appearance_term[k] = Compare_Text_with_terms_appearance(text, Dictionary_of_terms)
    
    for term in Dictionary_of_terms.Term:
       Filtered_DataFrame_out_filt['Appearance_term_'+term] = [Appearance_term[k][term.upper()] for k in Filtered_DataFrame_out_filt.id]

    
    return Filtered_DataFrame_out_filt
    
    

In [ ]:
def Semantic_search(Filtered_DataFrame_out_filt, Dictionary_of_terms):
    Dict_concepts_keys = {}
    Keys_list = ['english_abstract', 'primary_topic.display_name']
    Dict_concepts_keys = {}
    
    DF_with_key = {Filtered_DataFrame_out_filt.id[i]:';'.join([str(Filtered_DataFrame_out_filt[k][i]).upper() for k in Keys_list]) for i in Filtered_DataFrame_out_filt.index }

    Probability_term, Appearance_term = {}, {}
    for i,k in enumerate(DF_with_key.keys()):
        if i%1000==0:
            print(i)
        text = DF_with_key[k]
        Probability_term[k], Appearance_term[k] = Compare_Text_with_terms(text, Dictionary_of_terms)
    
    for term in Dictionary_of_terms.Term:
       Filtered_DataFrame_out_filt['Probability_term_'+term] = [Probability_term[k][term.upper()] for k in Filtered_DataFrame_out_filt.id]
       Filtered_DataFrame_out_filt['Appearance_term_'+term] = [Appearance_term[k][term.upper()] for k in Filtered_DataFrame_out_filt.id]

    
    return Filtered_DataFrame_out_filt
    
    

In [355]:
def make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms,semantic=True, prob_passing = 0.6):
    Dict_concept_region_papers = {}
    
    for concept in Dictionary_of_terms.Term:
        # if semantic:
        #     Journals_papers_concept = list(Filtered_DataFrame_out_filt[Filtered_DataFrame_out_filt['Probability_string_term_'+concept]=='>0.4']['primary_location.source.id'])
        # else:
        #     Journals_papers_concept = list(Filtered_DataFrame_out_filt[Filtered_DataFrame_out_filt['Appearance_term_'+concept]=='Yes']['primary_location.source.id'])

        Journals_papers_concept = Filtered_DataFrame_out_filt[(Filtered_DataFrame_out_filt['Probability_term_'+concept].apply(lambda x: max(x)>=prob_passing)) | (Filtered_DataFrame_out_filt['Appearance_term_'+concept]=='Yes')]['primary_location.source.id']
        # Journals_papers_concept = list(Filtered_DataFrame_out_filt[(Filtered_DataFrame_out_filt['Probability_term_'+concept].apply(lambda x: max(x)>=prob_passing))]['primary_location.source.id'])

        Dict_concept_region_papers[concept] = {}
        for journal in Journals_papers_concept:
            for region in Dict_countries_regions.keys():
                for country in Dict_countries_regions[region]:
                    if journal in Dict_journals_per_country[country]:
                        try:
                            Dict_concept_region_papers[concept][region] += [paper]
                        except:
                            Dict_concept_region_papers[concept][region] = [paper]
                            pass
    
    Dict_concept_region_number_of_papers = {}
    
    for k in Dict_concept_region_papers.keys():
        Dict_concept_region_number_of_papers[k] = {}
        for v in Dict_concept_region_papers[k].keys():
            n_papers_c_r = len(Dict_concept_region_papers[k][v])
            p_papers_r =  Dict_number_papers_per_region[v]
            Dict_concept_region_number_of_papers[k][v] = str(n_papers_c_r)+ ' (' + str(np.round(n_papers_c_r/p_papers_r,2)) + ')'
    
    return pd.DataFrame(Dict_concept_region_number_of_papers).sort_index()
    

In [462]:
Filtered_DataFrame_out_filt = read_pickle('Filtered_DataFrame_out_filt.pkl')
Filtered_DataFrame_out_filt = Filtered_DataFrame_out_filt.drop_duplicates('abstract')

In [476]:
%%time
file = 'Dict_of_terms/Dictionary_of_terms_Hegemonic_centric.csv'
Dictionary_of_terms = pd.read_table(file,delimiter=';',header=0)
Filtered_DataFrame_out_filt_1 = Semantic_search_fast(Filtered_DataFrame_out_filt, Dictionary_of_terms)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt_1, Dictionary_of_terms,prob_passing=0.65)
#It normally takes: minutes.

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
CPU times: user 26.8 s, sys: 332 ms, total: 27.1 s
Wall time: 27.2 s


,Security,Political order,Peacekeeping,State-building,Conflict resolution,Ceasefire,Democratic peace,Human rights,Peacebuilding,Governance,Reconciliation,Development
Australia and New Zealand,38 (0.31),1 (0.01),4 (0.03),2 (0.02),8 (0.07),3 (0.02),1 (0.01),17 (0.14),8 (0.07),7 (0.06),6 (0.05),36 (0.29)
Central Asia,8 (0.36),NaN,1 (0.05),1 (0.05),1 (0.05),NaN,NaN,NaN,1 (0.05),2 (0.09),1 (0.05),12 (0.55)
Eastern Asia,64 (0.11),NaN,3 (0.01),4 (0.01),9 (0.02),3 (0.01),1 (0.0),15 (0.03),23 (0.04),15 (0.03),16 (0.03),103 (0.18)
Eastern Europe,564 (0.41),11 (0.01),113 (0.08),20 (0.01),103 (0.07),12 (0.01),7 (0.01),140 (0.1),62 (0.04),75 (0.05),74 (0.05),445 (0.32)
Latin America and the Caribbean,402 (0.13),3 (0.0),59 (0.02),19 (0.01),513 (0.16),14 (0.0),4 (0.0),406 (0.13),195 (0.06),100 (0.03),230 (0.07),659 (0.21)
Micronesia,10 (0.67),NaN,2 (0.13),NaN,4 (0.27),NaN,NaN,3 (0.2),5 (0.33),3 (0.2),NaN,8 (0.53)
Northern Africa,31 (0.23),NaN,3 (0.02),NaN,2 (0.01),1 (0.01),2 (0.01),8 (0.06),3 (0.02),2 (0.01),5 (0.04),22 (0.16)
Northern America,669 (0.31),12 (0.01),71 (0.03),28 (0.01),201 (0.09),21 (0.01),25 (0.01),230 (0.11),193 (0.09),276 (0.13),134 (0.06),580 (0.27)
Northern Europe,856 (0.39),24 (0.01),77 (0.03),42 (0.02),159 (0.07),29 (0.01),23 (0.01),259 (0.12),306 (0.14),306 (0.14),140 (0.06),516 (0.23)
Polynesia,1 (0.11),NaN,NaN,NaN,3 (0.33),NaN,NaN,NaN,NaN,NaN,NaN,4 (0.44)


In [477]:
write_pickle(Filtered_DataFrame_out_filt, 'Filtered_DataFrame_out_filt_new.pkl')

In [436]:
%%time
file = 'Dict_of_terms/Dictionary_of_terms_Hegemonic_centric.csv'
Dictionary_of_terms = pd.read_table(file,delimiter=';',header=0)
# Filtered_DataFrame_out_filt = Semantic_search(Filtered_DataFrame_out_filt, Dictionary_of_terms)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms,prob_passing=0.65)
#It normally takes: minutes.

CPU times: user 6.8 s, sys: 422 ms, total: 7.22 s
Wall time: 7.3 s


,Security,Political order,Peacekeeping,State-building,Conflict resolution,Ceasefire,Democratic peace,Human rights,Peacebuilding,Governance,Reconciliation,Development
Australia and New Zealand,38 (0.31),1 (0.01),4 (0.03),3 (0.02),8 (0.07),3 (0.02),1 (0.01),17 (0.14),12 (0.1),7 (0.06),6 (0.05),43 (0.35)
Central Asia,9 (0.41),NaN,1 (0.05),1 (0.05),1 (0.05),NaN,NaN,NaN,1 (0.05),2 (0.09),1 (0.05),12 (0.55)
Eastern Asia,64 (0.11),NaN,3 (0.01),4 (0.01),9 (0.02),3 (0.01),1 (0.0),15 (0.03),26 (0.04),15 (0.03),17 (0.03),110 (0.19)
Eastern Europe,605 (0.44),11 (0.01),115 (0.08),20 (0.01),104 (0.08),13 (0.01),7 (0.01),140 (0.1),91 (0.07),75 (0.05),77 (0.06),468 (0.34)
Latin America and the Caribbean,419 (0.13),3 (0.0),61 (0.02),19 (0.01),513 (0.16),14 (0.0),4 (0.0),406 (0.13),226 (0.07),104 (0.03),237 (0.07),705 (0.22)
Micronesia,10 (0.67),NaN,2 (0.13),NaN,4 (0.27),NaN,NaN,3 (0.2),5 (0.33),3 (0.2),NaN,8 (0.53)
Northern Africa,31 (0.23),NaN,3 (0.02),NaN,2 (0.01),1 (0.01),2 (0.01),8 (0.06),5 (0.04),2 (0.01),5 (0.04),27 (0.2)
Northern America,699 (0.33),13 (0.01),73 (0.03),28 (0.01),202 (0.09),24 (0.01),26 (0.01),230 (0.11),286 (0.13),277 (0.13),135 (0.06),641 (0.3)
Northern Europe,881 (0.4),25 (0.01),79 (0.04),42 (0.02),163 (0.07),35 (0.02),23 (0.01),259 (0.12),487 (0.22),314 (0.14),148 (0.07),589 (0.27)
Polynesia,1 (0.11),NaN,NaN,NaN,3 (0.33),NaN,NaN,NaN,NaN,NaN,NaN,4 (0.44)


In [475]:
file = 'Dict_of_terms/Dictionary_of_terms_local_turn_critical_paradigms.csv'
Dictionary_of_terms_local_turn = pd.read_table(file,delimiter=';',header=0)
Filtered_DataFrame_out_filt = Semantic_search_fast(Filtered_DataFrame_out_filt, Dictionary_of_terms_local_turn)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms_local_turn,prob_passing=0.65)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000


,Liberal peace,Local turn,Everyday peace,Hybridity,Participation,Community resilience,Grassroots peace,Community-led initiatives
Australia and New Zealand,NaN,NaN,NaN,NaN,8 (0.07),2 (0.02),1 (0.01),NaN
Central Asia,1 (0.05),NaN,NaN,NaN,1 (0.05),NaN,NaN,NaN
Eastern Asia,1 (0.0),2 (0.0),2 (0.0),3 (0.01),11 (0.02),1 (0.0),1 (0.0),1 (0.0)
Eastern Europe,3 (0.0),4 (0.0),1 (0.0),2 (0.0),104 (0.08),NaN,NaN,1 (0.0)
Latin America and the Caribbean,21 (0.01),15 (0.0),1 (0.0),3 (0.0),315 (0.1),3 (0.0),NaN,7 (0.0)
Micronesia,NaN,1 (0.07),NaN,1 (0.07),2 (0.13),NaN,NaN,1 (0.07)
Northern Africa,NaN,NaN,NaN,NaN,3 (0.02),NaN,NaN,NaN
Northern America,21 (0.01),22 (0.01),7 (0.0),16 (0.01),112 (0.05),1 (0.0),2 (0.0),15 (0.01)
Northern Europe,54 (0.02),48 (0.02),19 (0.01),33 (0.01),155 (0.07),4 (0.0),3 (0.0),26 (0.01)
Polynesia,NaN,NaN,NaN,NaN,1 (0.11),NaN,NaN,NaN


In [ ]:
%%time
file = 'Dict_of_terms/Dictionary_of_terms_local_turn_critical_paradigms.csv'
Dictionary_of_terms_local_turn = pd.read_table(file,delimiter=';',header=0)
# Filtered_DataFrame_out_filt = Semantic_search(Filtered_DataFrame_out_filt, Dictionary_of_terms_local_turn)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms_local_turn,prob_passing=0.65)

CPU times: user 5.07 s, sys: 49.6 ms, total: 5.12 s
Wall time: 5.13 s


,Liberal peace,Local turn,Everyday peace,Hybridity,Participation,Community resilience,Grassroots peace,Community-led initiatives
Australia and New Zealand,1 (0.01),19 (0.15),NaN,13 (0.11),14 (0.11),4 (0.03),3 (0.02),12 (0.1)
Central Asia,2 (0.09),3 (0.14),NaN,4 (0.18),1 (0.05),NaN,NaN,1 (0.05)
Eastern Asia,3 (0.01),36 (0.06),2 (0.0),35 (0.06),26 (0.04),15 (0.03),4 (0.01),23 (0.04)
Eastern Europe,32 (0.02),190 (0.14),1 (0.0),205 (0.15),162 (0.12),37 (0.03),23 (0.02),150 (0.11)
Latin America and the Caribbean,130 (0.04),963 (0.3),12 (0.0),572 (0.18),582 (0.18),187 (0.06),138 (0.04),414 (0.13)
Micronesia,1 (0.07),7 (0.47),NaN,6 (0.4),4 (0.27),2 (0.13),1 (0.07),5 (0.33)
Northern Africa,1 (0.01),7 (0.05),NaN,7 (0.05),6 (0.04),3 (0.02),1 (0.01),3 (0.02)
Northern America,120 (0.06),495 (0.23),11 (0.01),456 (0.21),313 (0.15),213 (0.1),76 (0.04),383 (0.18)
Northern Europe,167 (0.08),672 (0.3),29 (0.01),640 (0.29),478 (0.22),311 (0.14),121 (0.05),538 (0.24)
Polynesia,NaN,1 (0.11),NaN,1 (0.11),1 (0.11),NaN,NaN,NaN


In [474]:
file = 'Dict_of_terms/Dictionary_of_terms_feminist_security_studies_critical_paradigms.csv'
Dictionary_of_terms_feminist_security_studies = pd.read_table(file,delimiter=';',header=0)

Filtered_DataFrame_out_filt = Semantic_search_fast(Filtered_DataFrame_out_filt, Dictionary_of_terms_feminist_security_studies)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms_feminist_security_studies,prob_passing=0.65)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000


,Feminist Peace,Care,Intersectionality,Continuum of violence
Australia and New Zealand,1 (0.01),11 (0.09),NaN,1 (0.01)
Eastern Asia,NaN,44 (0.08),NaN,1 (0.0)
Eastern Europe,3 (0.0),26 (0.02),1 (0.0),3 (0.0)
Latin America and the Caribbean,9 (0.0),88 (0.03),8 (0.0),4 (0.0)
Northern Africa,NaN,8 (0.06),NaN,NaN
Northern America,12 (0.01),114 (0.05),10 (0.0),13 (0.01)
Northern Europe,24 (0.01),76 (0.03),21 (0.01),18 (0.01)
South-eastern Asia,1 (0.0),40 (0.02),2 (0.0),2 (0.0)
Southern Asia,6 (0.01),57 (0.05),3 (0.0),2 (0.0)
Southern Europe,14 (0.01),17 (0.01),7 (0.01),13 (0.01)


In [ ]:
file = 'Dict_of_terms/Dictionary_of_terms_feminist_security_studies_critical_paradigms.csv'
Dictionary_of_terms_feminist_security_studies = pd.read_table(file,delimiter=';',header=0)

# Filtered_DataFrame_out_filt = Semantic_search(Filtered_DataFrame_out_filt, Dictionary_of_terms_feminist_security_studies)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms_feminist_security_studies,prob_passing=0.65)

,Feminist Peace,Care,Intersectionality,Continuum of violence
Australia and New Zealand,4 (0.03),18 (0.15),4 (0.03),1 (0.01)
Central Asia,NaN,1 (0.05),NaN,NaN
Eastern Asia,2 (0.0),60 (0.1),2 (0.0),1 (0.0)
Eastern Europe,8 (0.01),57 (0.04),7 (0.01),4 (0.0)
Latin America and the Caribbean,30 (0.01),165 (0.05),33 (0.01),5 (0.0)
Micronesia,NaN,1 (0.07),NaN,NaN
Northern Africa,NaN,9 (0.07),NaN,NaN
Northern America,29 (0.01),249 (0.12),27 (0.01),14 (0.01)
Northern Europe,92 (0.04),239 (0.11),92 (0.04),21 (0.01)
South-eastern Asia,8 (0.0),81 (0.04),8 (0.0),2 (0.0)


In [472]:
%%time
file = 'Dict_of_terms/Dictionary_of_terms_decolonial_thought_critical_paradigms.csv'
Dictionary_of_terms_decolonial_thought = pd.read_table(file,delimiter=';',header=0)
Filtered_DataFrame_out_filt_1 = Semantic_search_fast(Filtered_DataFrame_out_filt, Dictionary_of_terms_decolonial_thought)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt_1, Dictionary_of_terms_decolonial_thought,prob_passing=0.65)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
CPU times: user 5.52 s, sys: 632 ms, total: 6.15 s
Wall time: 6.26 s


,Self-determination,Self-government,Restorative justice,Pluriversal peace,Decolonial peace,Indigenous sovereignty
Australia and New Zealand,2 (0.02),NaN,1 (0.01),NaN,NaN,NaN
Central Asia,NaN,NaN,1 (0.05),NaN,NaN,NaN
Eastern Asia,1 (0.0),1 (0.0),NaN,NaN,NaN,NaN
Eastern Europe,17 (0.01),8 (0.01),4 (0.0),NaN,1 (0.0),NaN
Latin America and the Caribbean,13 (0.0),3 (0.0),67 (0.02),NaN,6 (0.0),NaN
Northern America,21 (0.01),2 (0.0),13 (0.01),NaN,1 (0.0),2 (0.0)
Northern Europe,26 (0.01),5 (0.0),12 (0.01),NaN,9 (0.0),NaN
South-eastern Asia,5 (0.0),5 (0.0),30 (0.01),1 (0.0),NaN,NaN
Southern Asia,15 (0.01),2 (0.0),5 (0.0),NaN,NaN,NaN
Southern Europe,12 (0.01),2 (0.0),10 (0.01),2 (0.0),1 (0.0),NaN


In [463]:
%%time
file = 'Dict_of_terms/Dictionary_of_terms_decolonial_thought_critical_paradigms.csv'
Dictionary_of_terms_decolonial_thought = pd.read_table(file,delimiter=';',header=0)
# Filtered_DataFrame_out_filt = Semantic_search(Filtered_DataFrame_out_filt, Dictionary_of_terms_decolonial_thought)
make_table_based_on_semantic_similarity(Filtered_DataFrame_out_filt, Dictionary_of_terms_decolonial_thought,prob_passing=0.65)

CPU times: user 358 ms, sys: 105 ms, total: 463 ms
Wall time: 507 ms


,Self-determination,Self-government,Restorative justice,Pluriversal peace,Decolonial peace,Indigenous sovereignty
Australia and New Zealand,2 (0.02),5 (0.04),1 (0.01),NaN,NaN,NaN
Central Asia,NaN,NaN,1 (0.05),NaN,NaN,NaN
Eastern Asia,2 (0.0),5 (0.01),NaN,NaN,NaN,NaN
Eastern Europe,18 (0.01),17 (0.01),5 (0.0),NaN,2 (0.0),1 (0.0)
Latin America and the Caribbean,13 (0.0),30 (0.01),81 (0.03),NaN,29 (0.01),2 (0.0)
Northern Africa,NaN,2 (0.01),NaN,NaN,NaN,NaN
Northern America,23 (0.01),62 (0.03),16 (0.01),NaN,12 (0.01),4 (0.0)
Northern Europe,28 (0.01),57 (0.03),18 (0.01),1 (0.0),25 (0.01),4 (0.0)
South-eastern Asia,6 (0.0),18 (0.01),30 (0.01),1 (0.0),2 (0.0),1 (0.0)
Southern Asia,16 (0.01),33 (0.03),5 (0.0),NaN,1 (0.0),NaN


In [487]:
concept = 'Security'
Smaller_DF_concept = Filtered_DataFrame_out_filt_1[(Filtered_DataFrame_out_filt_1['Probability_term_'+concept].apply(lambda x: x[0]>=prob_passing)) | (Filtered_DataFrame_out_filt_1['Appearance_term_'+concept]=='Yes')]
DF_with_country = Smaller_DF_concept[(~Smaller_DF_concept['authorships.countries'].isnull())]

Countries_aff_to_check = [i for i in set([j for i in DF_with_country['authorships.countries'] for j in i.split('|')])][1:]
Countries_aff_to_check

['KW',
 'CY',
 'ES',
 'SE',
 'ID',
 'YE',
 'MT',
 'AS',
 'AR',
 'EG',
 'LT',
 'DZ',
 'FJ',
 'AM',
 'BO',
 'MM',
 'GE',
 'MK',
 'KH',
 'UA',
 'RU',
 'AZ',
 'CH',
 'AO',
 'GM',
 'KR',
 'ML',
 'AU',
 'QA',
 'BT',
 'ET',
 'LB',
 'HU',
 'TM',
 'AF',
 'BY',
 'NZ',
 'JO',
 'TZ',
 'MC',
 'UG',
 'IE',
 'SN',
 'US',
 'ZW',
 'GB',
 'IN',
 'KZ',
 'TW',
 'BR',
 'PK',
 'TL',
 'BD',
 'RE',
 'PT',
 'WS',
 'SI',
 'PH',
 'CG',
 'EE',
 'ZM',
 'BH',
 'PG',
 'VE',
 'FR',
 'RS',
 'AL',
 'FI',
 'TN',
 'TH',
 'CD',
 'SY',
 'MY',
 'IR',
 'SA',
 'SV',
 'LR',
 'MW',
 'PY',
 'BG',
 'JP',
 'MX',
 'PS',
 'MG',
 'KG',
 'BN',
 'SG',
 'UZ',
 'RW',
 'AE',
 'UY',
 'IL',
 'HK',
 'MR',
 'PL',
 'PA',
 'GR',
 'NL',
 'CI',
 'TT',
 'TJ',
 'CA',
 'CF',
 'DK',
 'MD',
 'NP',
 'CM',
 'CL',
 'XK',
 'AT',
 'IQ',
 'RO',
 'LV',
 'IT',
 'SK',
 'SS',
 'LU',
 'VN',
 'SL',
 'OM',
 'NG',
 'IS',
 'SO',
 'BJ',
 'LK',
 'HN',
 'PE',
 'KE',
 'ZA',
 'CR',
 'SD',
 'MA',
 'CO',
 'GH',
 'TR',
 'NO',
 'DE',
 'BI',
 'BZ',
 'CN',
 'BE',
 'MZ',
 'CZ',

In [ ]:
# %%time
def create_dict_for_pie_chart_country(Filtered_DataFrame_out_filt, concepts,Country_codes, prob_passing):
    Dict_countries_mentioned = {}
    for concept in concepts:
        Smaller_DF_concept = Filtered_DataFrame_out_filt[(Filtered_DataFrame_out_filt['Probability_term_'+concept].apply(lambda x: x[0]>=prob_passing)) | (Filtered_DataFrame_out_filt['Appearance_term_'+concept]=='Yes')]
        DF_with_country = Smaller_DF_concept[(~Smaller_DF_concept['authorships.countries'].isnull())]
        
        Countries_aff_to_check = [i for i in set([j for i in DF_with_country['authorships.countries'] for j in i.split('|')])][1:]
        # Filtered_countries = Country_codes.iloc[[i for i in Country_codes.index if i not in [25,32]]]
        list_countries = [i for i in Country_codes.Country]
        
        for concept_0 in Countries_aff_to_check:
            print(concept_0)
            for concept_2 in list_countries:
                for i in DF_with_country.index:
                    abstract = Smaller_DF_concept['english_abstract'][i].upper()
                    id_paper = Smaller_DF_concept['id'][i]
                    try:
                        if concept_0 in DF_with_country['authorships.countries'][i]:
                            if concept_2.upper() in abstract:
                                if concept_2 == 'NIGER' or concept_2 == 'MALI':
                                    if concept_2.upper()+' ' in abstract or concept_2.upper()+',' or concept_2.upper()+'.':
                                        try:
                                            Dict_countries_mentioned[(concept_0,concept,concept_2)] += [id_paper]
                                        except:
                                            Dict_countries_mentioned[(concept_0,concept,concept_2)] = [id_paper]   
                                            pass

                                else:
                                    try:
                                        Dict_countries_mentioned[(concept_0,concept,concept_2)] += [id_paper]
                                    except:
                                        Dict_countries_mentioned[(concept_0,concept,concept_2)] = [id_paper]   
                                        pass
                    except:
                        pass
    write_pickle(Dict_countries_mentioned, 'Dict_countries_mentioned/Dict_countries_mentioned_'+concept+'.pkl')
    # return Dict_countries_mentioned

terms = {'Hegemonic':['Security', 'Political order', 'Peacekeeping', 'State-building', 'Conflict resolution', 'Ceasefire',
                      'Democratic peace', 'Human rights', 'Peacebuilding', 'Governance', 'Reconciliation', 'Development'],
        'Local turn':['Liberal peace', 'Local turn', 'Everyday peace', 'Hybridity', 'Participation', 'Community resilience', 'Grassroots peace', 'Community-led initiatives'],
         'Feminist security':['Feminist Peace', 'Care', 'Intersectionality', 'Continuum of violence'],
         'Decolonial thought':['Self-determination', 'Self-government','Restorative justice', 'Pluriversal peace',
                               'Decolonial peace', 'Indigenous sovereignty']
}

prob_passing = 0.65
# for k in terms.keys():
k= 'Local turn'
concepts = terms[k]
create_dict_for_pie_chart_country(Filtered_DataFrame_out_filt, concepts,Country_codes, prob_passing)

# Dict_region_per_country = {c:u for u,v in Dict_countries_regions.items() for c in v}

KW
CY
ES
SE
ID
YE
MT
AS
AR
EG
LT
DZ
FJ
AM
BO
MM
GE
MK
KH
UA
RU
AZ
CH
AO
GM
KR
ML
AU
QA
BT
ET
LB
HU
TM
AF
BY
NZ
JO
TZ
MC
UG
IE
SN
US
ZW
GB
IN
KZ
TW
BR
PK
TL
BD
RE
PT
WS
SI
PH
CG
EE
ZM
BH
PG
VE
FR
RS
AL
FI
TN
TH
CD
SY
MY
IR
SA
SV
LR
MW
PY
BG
JP
MX
PS
MG
KG
BN
SG
UZ
RW
AE
UY
IL
HK
MR
PL
PA
GR
NL
CI
TT
TJ
CA
CF
DK
MD
NP
CM
CL
XK
AT
IQ
RO
LV
IT
SK
SS
LU
VN
SL
OM
NG
IS
SO
BJ
LK
HN
PE
KE
ZA
CR
SD
MA
CO
GH
TR
NO
DE
BI
BZ
CN
BE
MZ
CZ
EC
HR
BA
ES
AT
SE
JP
ID
RO
MX
IT
UG
US
GB
BR
GE
SL
NG
MK
UA
PL
CH
PE
EE
GR
KE
ZA
NL
FR
RS
AU
CO
NO
DE
CD
HU
CN
BE
IR
CZ
CA
DK
MD
BA
CY
ES
MN
SE
ID
YE
AR
LT
DZ
AM
MM
MK
UA
RU
CH
GM
KR
AU
BT
ET
HU
TM
AF
JO
TZ
UG
IE
SN
US
GB
IN
KZ
TW
BR
PK
BD
PT
SI
PH
CG
ZM
FR
RS
VE
FI
TH
CD
MY
IR
JP
MX
BN
SG
UZ
UY
IL
PL
GR
NL
CI
TT
TJ
CA
DK
NP
CM
CL
XK
AT
RO
IT
LU
NG
GA
SO
LK
PE
KE
ZA
CR
CO
GH
TR
NO
DE
BI
CN
BE
MZ
CZ
EC
HR
XK
ES
AT
SE
JP
ID
MT
IQ
LV
MX
IT
UG
SN
US
SS
GB
TW
IN
AM
BR
BO
PK
NG
MK
UA
RU
PL
CH
PT
SO
LK
GR
KE
ZA
FR
RS
AU
CO
FI
NO
TR
DE
QA
ET
MY
HU
CN
BE
AF
CZ
DK
NZ
PF
BA
C

In [500]:
# %%time
import plotly.express as px 
def generate_pie_charts(Dict_countries_mentioned, concept,name, color_number, region=False):
    
    import plotly.express as px 
    # key='abstract'
    DF_Hierarchies = {}
    if region:
        DF_Hierarchies['Host']= [i[0][:-9] for i,j in Dict_countries_mentioned.items()]
        DF_Hierarchies['Nation'] = [i[2][:-9] for i,j in Dict_countries_mentioned.items()]
    else:
        DF_Hierarchies['Host']= [i[0] for i,j in Dict_countries_mentioned.items()]
        DF_Hierarchies['Nation'] = [i[2] for i,j in Dict_countries_mentioned.items()]

    DF_Hierarchies['Concept'] = [ i[1] for i,j in Dict_countries_mentioned.items()]
    DF_Hierarchies['Mentions'] = [len(j) for i,j in Dict_countries_mentioned.items()]
    DF_Hierarchies = pd.DataFrame(DF_Hierarchies)
    DF_Hierarchies = DF_Hierarchies[DF_Hierarchies.Concept == concept]
    
    Colors_concepts = ['rgba(233, 217, 133, 1)','rgba(178, 189, 126, 1)', 'rgba(116, 156, 117, 1)', 'rgba(106, 93, 123, 1)', '#664A51', '#A54F4F', '#2660A4','#C47335','#8E427B','rgba(233, 217, 133, 1)','rgba(178, 189, 126, 1)', 'rgba(116, 156, 117, 1)']
    
    fig2 = px.sunburst(DF_Hierarchies, path=['Concept', 'Nation', 'Host'], values='Mentions', color='Concept', color_discrete_sequence=[Colors_concepts[color_number]])
    # fig2.update_layout(title_text="Two-level Sunburst Diagram", font_size=10)
    fig2.update_layout(font_size=12, autosize=False,width=800,height=1200)
    fig2.write_html("Hierarchy_concept/Hierarchy_concept_host_nation_"+concept+"_new.html")

    # return DF_Hierarchies
    

In [540]:
concept = 'Indigenous sovereignty'
Smaller_DF_concept = Filtered_DataFrame_out_filt[(Filtered_DataFrame_out_filt['Probability_term_'+concept].apply(lambda x: x[0]>=prob_passing)) | (Filtered_DataFrame_out_filt['Appearance_term_'+concept]=='Yes')]
DF_with_country = Smaller_DF_concept[(~Smaller_DF_concept['authorships.countries'].isnull())]

Countries_aff_to_check = [i for i in set([j for i in DF_with_country['authorships.countries'] for j in i.split('|')])][1:]
# Filtered_countries = Country_codes.iloc[[i for i in Country_codes.index if i not in [25,32]]]
list_countries = [i for i in Country_codes.Country]

Dict_countries_mentioned = {}
for concept_0 in Countries_aff_to_check:
    print(concept_0)
    for concept_2 in list_countries:
        for i in DF_with_country.index:
            abstract = Smaller_DF_concept['english_abstract'][i].upper()
            id_paper = Smaller_DF_concept['id'][i]
            try:
                if concept_0 in DF_with_country['authorships.countries'][i]:
                    if concept_2.upper() in abstract:
                        if concept_2 == 'NIGER' or concept_2 == 'MALI':
                            if concept_2.upper()+' ' in abstract or concept_2.upper()+',' or concept_2.upper()+'.':
                                try:
                                    Dict_countries_mentioned[(concept_0,concept,concept_2)] += [id_paper]
                                except:
                                    Dict_countries_mentioned[(concept_0,concept,concept_2)] = [id_paper]   
                                    pass

                        else:
                            try:
                                Dict_countries_mentioned[(concept_0,concept,concept_2)] += [id_paper]
                            except:
                                Dict_countries_mentioned[(concept_0,concept,concept_2)] = [id_paper]   
                                pass
            except:
                pass

In [543]:
Smaller_DF_concept

,id,abstract,authorships.author.display_name,authorships.author.id,authorships.author.orcid,authorships.countries,authorships.is_corresponding,best_oa_location.license,cited_by_count,corresponding_institution_ids,...,Appearance_term_Continuum of violence,Probability_term_Pluriversal peace,Probability_string_term_Pluriversal peace,Appearance_term_Pluriversal peace,Probability_term_Decolonial peace,Probability_string_term_Decolonial peace,Appearance_term_Decolonial peace,Probability_term_Feminist Peace,Probability_string_term_Feminist Peace,Appearance_term_Feminist Peace
89049,https://openalex.org/W4417136364,The Book of Mormon's portrayal of a great peac...,Thomas Murphy,https://openalex.org/A5042984994,https://orcid.org/0000-0002-8876-6099,NaN,True,NaN,0,NaN,...,No,[0.16],<=0.4,No,[0.2],<=0.4,No,[0.19],<=0.4,No
90611,https://openalex.org/W7133309692,Two hundred and fifty years after the American...,John Gilbert McCurdy,https://openalex.org/A5113879253,NaN,US,True,NaN,0,https://openalex.org/I138216421,...,No,[0.15],<=0.4,No,[0.29],<=0.4,No,[0.16],<=0.4,No


In [ ]:
# dict_files = ['Dict_countries_mentioned/Dict_countries_mentioned_Development.pkl',
                'Dict_countries_mentioned/Dict_countries_mentioned_Community-led initiatives.pkl'
#               'Dict_countries_mentioned/Dict_countries_mentioned_Continuum of violence.pkl',
#               'Dict_countries_mentioned/Dict_countries_mentioned_Indigenous sovereignty.pkl']

# Dict_countries_mentioned = {}
# for file in dict_files:
#     temporal_dict = read_pickle(file)
#     Dict_countries_mentioned.update(temporal_dict)


# print(len(Dict_countries_mentioned))
Dict_countries_mentioned = {k:v for k,v in Dict_countries_mentioned.items() if 'Mali' not in k}
# print(len(Dict_countries_mentioned))
# for term,concepts in terms.items():
#     concept 
print('-----------')
# for color_number, concept in enumerate(concepts):
concept = 'Everyday peace'

generate_pie_charts(Dict_countries_mentioned,concept, '_country',2)
print(concept)


# for color_number, concept in enumerate(concepts):
#     print(concept)
#     generate_pie_charts(Dict_regions_mentioned,concept, name+'_region',color_number)
# Dict_countries_mentioned-
len(Dict_countries_mentioned)

-----------
Everyday peace


86

In [539]:
dict_files = ['Dict_countries_mentioned/Dict_countries_mentioned_Development.pkl',
              'Dict_countries_mentioned/Dict_countries_mentioned_Community-led initiatives.pkl',
              'Dict_countries_mentioned/Dict_countries_mentioned_Continuum of violence.pkl',
              'Dict_countries_mentioned/Dict_countries_mentioned_Indigenous sovereignty.pkl']

Dict_countries_mentioned = {}
for file in dict_files:
    temporal_dict = read_pickle(file)
    Dict_countries_mentioned.update(temporal_dict)


print(len(Dict_countries_mentioned))
Dict_countries_mentioned = {k:v for k,v in Dict_countries_mentioned.items() if 'Mali' not in k}
print(len(Dict_countries_mentioned))
for term,concepts in terms.items():
    print('-----------')
    for color_number, concept in enumerate(concepts):
        
        try:
            generate_pie_charts(Dict_countries_mentioned,concept, '_country',color_number)
            print(concept)
        except:
            pass

# for color_number, concept in enumerate(concepts):
#     print(concept)
#     generate_pie_charts(Dict_regions_mentioned,concept, name+'_region',color_number)
# Dict_countries_mentioned-
len(Dict_countries_mentioned)

17293
16756
-----------
Security
Political order
Peacekeeping
State-building
Conflict resolution
Ceasefire
Democratic peace
Human rights
Peacebuilding
Governance
Reconciliation
Development
-----------
Liberal peace
Local turn
Everyday peace
Hybridity
Participation
Community resilience
Grassroots peace
Community-led initiatives
-----------
Feminist Peace
Care
Intersectionality
Continuum of violence
-----------
Self-determination
Self-government
Restorative justice
Pluriversal peace
Decolonial peace
Indigenous sovereignty


16756

In [ ]:
# generate_pie_charts(Dict_countries_mentioned,concept, name+'_country',color_number)

DF_Hierarchies = {}
if region:
    DF_Hierarchies['Host']= [i[0][:-9] for i,j in Dict_countries_mentioned.items()]
    DF_Hierarchies['Nation'] = [i[2][:-9] for i,j in Dict_countries_mentioned.items()]
else:
    DF_Hierarchies['Host']= [i[0] for i,j in Dict_countries_mentioned.items()]
    DF_Hierarchies['Nation'] = [i[2] for i,j in Dict_countries_mentioned.items()]

DF_Hierarchies['Concept'] = [ i[1] for i,j in Dict_countries_mentioned.items()]
DF_Hierarchies['Mentions'] = [len(j) for i,j in Dict_countries_mentioned.items()]
DF_Hierarchies = pd.DataFrame(DF_Hierarchies)
DF_Hierarchies = DF_Hierarchies[DF_Hierarchies.Concept == concept]
DF_Hierarchies.Host.value_counts()

Host
    1571
Name: count, dtype: int64

In [200]:
Dict_countries_mentioned.keys()

dict_keys([('GM', 'Security', 'Angola'), ('GM', 'Security', 'China'), ('GM', 'Security', 'France'), ('GM', 'Security', 'Gambia'), ('GM', 'Security', 'Mozambique'), ('GM', 'Security', 'Russia'), ('GM', 'Security', 'South Africa'), ('GM', 'Security', 'United Kingdom'), ('GM', 'Security', 'United States'), ('TM', 'Security', 'Turkmenistan'), ('NZ', 'Security', 'Afghanistan'), ('NZ', 'Security', 'Guinea'), ('NZ', 'Security', 'Ireland'), ('NZ', 'Security', 'New Zealand'), ('NZ', 'Security', 'Papua New Guinea'), ('MK', 'Security', 'Albania'), ('MK', 'Security', 'Australia'), ('MK', 'Security', 'Bosnia and Herzegovina'), ('MK', 'Security', 'Congo'), ('MK', 'Security', 'Croatia'), ('MK', 'Security', 'Macedonia'), ('MK', 'Security', 'Montenegro'), ('MK', 'Security', 'Russia'), ('MK', 'Security', 'Serbia'), ('MK', 'Security', 'Turkey'), ('MK', 'Security', 'Ukraine'), ('IT', 'Security', 'Afghanistan'), ('IT', 'Security', 'Algeria'), ('IT', 'Security', 'Armenia'), ('IT', 'Security', 'Australia'), 

In [191]:

for color_number, concept in enumerate(concepts):
    print(concept)
    generate_pie_charts(Dict_regions_mentioned,concept, name+'_region',color_number)


Security


NameError: name 'Dict_regions_mentioned' is not defined

In [192]:
def Generate_plot_over_under_studied(concept, DF_Hierarchies,Country_codes, max_y, max_x,all_countries=False):

    DF_Hierarchies['Nation_code']  = [Country_codes[Country_codes.Country == nation].iloc[0]['Alpha-2 code'].replace(' ','') for nation in DF_Hierarchies['Nation']]
    
    Host_countries = list(set(DF_Hierarchies['Host']))
    Nation_countries = list(set(DF_Hierarchies['Nation_code']))
    
    Dict_host_country_same_country = {}
    total_mentions_done = {}
    for host in Host_countries:
        host_df = DF_Hierarchies[DF_Hierarchies.Host == host]
        total_mentions_done[host] = host_df.Mentions.sum()
        try:
            self_mentions = host_df[DF_Hierarchies.Nation_code == host].iloc[0].Mentions
        except:
            self_mentions = 0
            pass
        Dict_host_country_same_country[host] = self_mentions/total_mentions_done[host]
    
    Nation_countries = list(set(DF_Hierarchies['Nation_code']))
    
    Dict_same_country_received = {}
    total_mentions_received = {}
    for nation in Host_countries:
        nation_df = DF_Hierarchies[DF_Hierarchies.Nation_code == nation]
        total_mentions_received[nation] = nation_df.Mentions.sum()
        try:
            self_mentions = host_df[DF_Hierarchies.Host == host].iloc[0].Mentions
        except:
            self_mentions = 0
            pass
        Dict_same_country_received[nation] = self_mentions/total_mentions_received[nation]
    
    
    
    Colors = ['#0496FF','#FFBC42','#D81159','#8F2D56']
    ncols = 1
    nrows = 1
    font_size_out = 20
    
    fig, axes = plt.subplots(ncols=ncols, nrows=nrows,figsize=(5*ncols,4*nrows), sharey=True)
    
    for region in colors_regions.keys():
        x_values = [i for k,i in total_mentions_done.items() if k in Dict_countries_regions[region]]
        y_values = [i for k,i in total_mentions_received.items() if k in Dict_countries_regions[region]]
        countries_plot = [k for k,i in total_mentions_received.items() if k in Dict_countries_regions[region]]
    
        colors_scatter = ['purple' for k,i in total_mentions_received.items() if k in Dict_countries_regions[region]]
    
        plt.plot((0, max_x), (0, max_y), label="Line 1", linestyle='--')
        
        plt.scatter(x_values, y_values, color = colors_scatter, label = region, alpha=0.5)

        
        for i, x in enumerate(x_values):
            if all_countries:
                plt.text(x,y_values[i], countries_plot[i])
            else:
                if x > max_y:
                    plt.text(x,y_values[i], countries_plot[i])
                
    
        for i, y in enumerate(y_values):
            if y > max_x/6:
                plt.text(x_values[i],y, countries_plot[i])
    
    axes.tick_params( 'x', size = 7, colors="black", labelsize=font_size_out-5)
    axes.tick_params( 'y', size = 7, colors="black", labelsize=font_size_out-5)
    # axes.set_title(i, size=font_size_out+2, weight='bold')
    
    axes.set_xlabel('Number of mentions given', fontsize=font_size_out-5)
    axes.set_ylabel('Number of mentions received', fontsize=font_size_out-5, labelpad=10, va='center', ha='center')
    axes.spines['right'].set_visible(False)
    axes.spines['top'].set_visible(False)
    axes.yaxis.set_ticks_position('left')
    axes.xaxis.set_ticks_position('bottom')
    
    axes.xaxis.set_tick_params(width=2)
    axes.yaxis.set_tick_params(width=2)
    
    for axis in ['top','bottom','left','right']:
        axes.spines[axis].set_linewidth(2)
        axes.spines[axis].set_color("black")
        axes.spines[axis].set_zorder(0)
    plt.savefig("Scatter_mentions_"+concept+".pdf", bbox_inches='tight')
    plt.show()
        